GridWorld Environment   

In [ ]:
# !pip install copier


Defaulting to user installation because normal site-packages is not writeable


In [36]:
# !pip install gymnasium numpy

import numpy as np
import gymnasium as gym
from typing import Optional

print("gymnasium version:", gym.__version__)
print("numpy version:", np.__version__)

gymnasium version: 1.3.0
numpy version: 2.1.2


In [37]:
class GridWorldEnv(gym.Env):

    def __init__(self, size: int = 5):
        # The size of the square grid (5x5 by default)
        self.size = size

        # Initialize positions - will be set randomly in reset()
        # Using -1,-1 as "uninitialized" state
        self._agent_location = np.array([-1, -1], dtype=np.int32)
        self._target_location = np.array([-1, -1], dtype=np.int32)

        # Define what the agent can observe
        # Dict space gives us structured, human-readable observations
        self.observation_space = gym.spaces.Dict(
            {
                "agent": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),   # [x, y] coordinates
                "target": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),  # [x, y] coordinates
            }
        )

        # Define what actions are available (4 directions)
        self.action_space = gym.spaces.Discrete(4)

        # Map action numbers to actual movements on the grid
        # This makes the code more readable than using raw numbers
        self._action_to_direction = {
            0: np.array([0, 1]),   # Move right (column + 1)
            1: np.array([-1, 0]),  # Move up (row - 1)
            2: np.array([0, -1]),  # Move left (column - 1)
            3: np.array([1, 0]),   # Move down (row + 1)
        }

    def _get_obs(self):
        """Convert internal state to observation format.

        Returns:
            dict: Observation with agent and target positions
        """
        return {"agent": self._agent_location, "target": self._target_location}    

    def _get_info(self):
        """Compute auxiliary information for debugging.

        Returns:
            dict: Info with distance between agent and target
        """
        return {
            "distance": np.linalg.norm(
                self._agent_location - self._target_location, ord=1
            )
        }

    
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        """Start a new episode.

        Args:
            seed: Random seed for reproducible episodes
            options: Additional configuration (unused in this example)

        Returns:
            tuple: (observation, info) for the initial state
        """
        # IMPORTANT: Must call this first to seed the random number generator
        super().reset(seed=seed)

        # Randomly place the agent anywhere on the grid
        self._agent_location = self.np_random.integers(0, self.size, size=2, dtype=int)

        # Randomly place target, ensuring it's different from agent position
        self._target_location = self._agent_location
        while np.array_equal(self._target_location, self._agent_location):
            self._target_location = self.np_random.integers(
                0, self.size, size=2, dtype=int
            )

        observation = self._get_obs()
        info = self._get_info()

        return observation, info


    def step(self, action):
        """Execute one timestep within the environment.

        Args:
            action: The action to take (0-3 for directions)

        Returns:
            tuple: (observation, reward, terminated, truncated, info)
        """
        # Map the discrete action (0-3) to a movement direction
        direction = self._action_to_direction[action]

        # Update agent position, ensuring it stays within grid bounds
        # np.clip prevents the agent from walking off the edge
        self._agent_location = np.clip(
            self._agent_location + direction, 0, self.size - 1
        )

        # Check if agent reached the target
        terminated = np.array_equal(self._agent_location, self._target_location)

        # We don't use truncation in this simple environment
        # (could add a step limit here if desired)
        truncated = False

        # Simple reward structure: +1 for reaching target, 0 otherwise
        # Alternative: could give small negative rewards for each step to encourage efficiency
        reward = 1 if terminated else 0

        observation = self._get_obs()
        info = self._get_info()

        return observation, reward, terminated, truncated, info

In [38]:
env = GridWorldEnv(size=5)

obs, info = env.reset(seed=42)
print("Initial observation:", obs)
print("Initial info:", info)

obs, reward, terminated, truncated, info = env.step(0)  # move right
print("\nAfter one step (action=0, 'right'):")
print("obs:", obs, "reward:", reward, "terminated:", terminated, "info:", info)

Initial observation: {'agent': array([0, 3]), 'target': array([3, 2])}
Initial info: {'distance': np.float64(4.0)}

After one step (action=0, 'right'):
obs: {'agent': array([0, 4]), 'target': array([3, 2])} reward: 0 terminated: False info: {'distance': np.float64(5.0)}


Registering the environment

In [39]:
# Register the environment so we can create it with gym.make()
gym.register(
    id="gymnasium_env/GridWorld-v0",
    entry_point=GridWorldEnv,
    max_episode_steps=300,  # Prevent infinite episodes
)
print("Registered!")
gym.pprint_registry()

Registered!
===== classic_control =====
Acrobot-v1             CartPole-v0            CartPole-v1
MountainCar-v0         MountainCarContinuous-v0 Pendulum-v1
===== phys2d =====
phys2d/CartPole-v0     phys2d/CartPole-v1     phys2d/Pendulum-v0
===== box2d =====
BipedalWalker-v3       BipedalWalkerHardcore-v3 CarRacing-v3
LunarLander-v3         LunarLanderContinuous-v3
===== toy_text =====
Blackjack-v1           CliffWalking-v1        CliffWalkingSlippery-v1
FrozenLake-v1          FrozenLake8x8-v1       Taxi-v4
===== tabular =====
tabular/Blackjack-v0   tabular/CliffWalking-v0
===== None =====
Ant-v2                 Ant-v3                 GymV21Environment-v0
GymV26Environment-v0   HalfCheetah-v2         HalfCheetah-v3
Hopper-v2              Hopper-v3              Humanoid-v2
Humanoid-v3            HumanoidStandup-v2     InvertedDoublePendulum-v2
InvertedPendulum-v2    Pusher-v2              Reacher-v2
Swimmer-v2             Swimmer-v3             Walker2d-v2
Walker2d-v3
===== mujoco ====

In [40]:
env = gym.make("gymnasium_env/GridWorld-v0", size=5)
print(env)

# Access the unwrapped instance to reach your custom attributes directly
print("Grid size:", env.unwrapped.size)

<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridWorldEnv<gymnasium_env/GridWorld-v0>>>>>
Grid size: 5


In [41]:
# Test specific action sequences to verify behavior
env = gym.make("gymnasium_env/GridWorld-v0")
obs, info = env.reset(seed=42)  # Use seed for reproducible testing

print(f"Starting position - Agent: {obs['agent']}, Target: {obs['target']}")

# Test each action type
actions = [0, 1, 2, 3]  # right, up, left, down
for action in actions:
    old_pos = obs['agent'].copy()
    obs, reward, terminated, truncated, info = env.step(action)
    new_pos = obs['agent']
    print(f"Action {action}: {old_pos} -> {new_pos}, reward={reward}")


Starting position - Agent: [0 3], Target: [3 2]
Action 0: [0 3] -> [0 4], reward=0
Action 1: [0 4] -> [0 4], reward=0
Action 2: [0 4] -> [0 3], reward=0
Action 3: [0 3] -> [1 3], reward=0


Check Environment Validity

In [42]:
from gymnasium.utils.env_checker import check_env

# This will catch many common issues
try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print(f"Environment has issues: {e}")

Environment passes all checks!


Using Wrappers

In [43]:
from gymnasium.wrappers import FlattenObservation

env = gym.make("gymnasium_env/GridWorld-v0")
print("Original observation_space:", env.observation_space)

wrapped_env = FlattenObservation(env)
print("Flattened observation_space:", wrapped_env.observation_space)

obs, info = wrapped_env.reset(seed=0)
print("Flattened obs (agent_x, agent_y, target_x, target_y):", obs)

Original observation_space: Dict('agent': Box(0, 4, (2,), int64), 'target': Box(0, 4, (2,), int64))
Flattened observation_space: Box(0, 4, (4,), int64)
Flattened obs (agent_x, agent_y, target_x, target_y): [4 3 2 1]


In [44]:
vec_env = gym.make_vec("gymnasium_env/GridWorld-v0", num_envs=3)
print(vec_env)

obs, info = vec_env.reset(seed=0)
print("Batched obs:", obs)

SyncVectorEnv(gymnasium_env/GridWorld-v0, num_envs=3)
Batched obs: {'agent': array([[4, 3],
       [2, 2],
       [4, 1]]), 'target': array([[2, 1],
       [3, 4],
       [0, 1]])}


Adding Rendering

In [45]:
class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 4}

    def __init__(self, size: int = 5, render_mode: Optional[str] = None):
        self.size = size
        self.render_mode = render_mode

        self._agent_location = np.array([-1, -1], dtype=np.int32)
        self._target_location = np.array([-1, -1], dtype=np.int32)

        self.observation_space = gym.spaces.Dict(
            {
                "agent": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),
                "target": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),
            }
        )

        self.action_space = gym.spaces.Discrete(4)

        self._action_to_direction = {
            0: np.array([0, 1]),   # right
            1: np.array([-1, 0]),  # up
            2: np.array([0, -1]),  # left
            3: np.array([1, 0]),   # down
        }

    def _get_obs(self):
        """Convert internal state into the observation format."""
        return {"agent": self._agent_location, "target": self._target_location}

    def _get_info(self):
        """Compute auxiliary debug info (Manhattan distance)."""
        return {
            "distance": np.linalg.norm(
                self._agent_location - self._target_location, ord=1
            )
        }

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        # IMPORTANT: must call this first — it seeds self.np_random
        super().reset(seed=seed)

        # Randomly place the agent anywhere on the grid
        self._agent_location = self.np_random.integers(0, self.size, size=2, dtype=int)

        # Randomly place the target, ensuring it differs from the agent
        self._target_location = self._agent_location
        while np.array_equal(self._target_location, self._agent_location):
            self._target_location = self.np_random.integers(
                0, self.size, size=2, dtype=int
            )

        observation = self._get_obs()
        info = self._get_info()

        return observation, info

    def step(self, action):
        # Map the discrete action (0-3) to a movement direction
        direction = self._action_to_direction[action]

        # Update position, clipping to stay within grid bounds
        self._agent_location = np.clip(
            self._agent_location + direction, 0, self.size - 1
        )

        # Episode ends when the agent reaches the target
        terminated = np.array_equal(self._agent_location, self._target_location)
        truncated = False  # no step-limit logic here (max_episode_steps handles it at registration)
        reward = 1 if terminated else 0

        observation = self._get_obs()
        info = self._get_info()

        return observation, reward, terminated, truncated, info

    def render(self):
        """Simple ASCII rendering: A = agent, T = target."""
        if self.render_mode == "human":
            for y in range(self.size - 1, -1, -1):  # print top row first
                row = ""
                for x in range(self.size):
                    if np.array_equal([x, y], self._agent_location):
                        row += "A "
                    elif np.array_equal([x, y], self._target_location):
                        row += "T "
                    else:
                        row += ". "
                print(row)
            print()

In [46]:
gym.register(
    id="gymnasium_env/GridWorld-v0",
    entry_point=GridWorldEnv,
    max_episode_steps=300,  # prevents infinite episodes
)

print("Registered!")
gym.pprint_registry()

Registered!
===== classic_control =====
Acrobot-v1             CartPole-v0            CartPole-v1
MountainCar-v0         MountainCarContinuous-v0 Pendulum-v1
===== phys2d =====
phys2d/CartPole-v0     phys2d/CartPole-v1     phys2d/Pendulum-v0
===== box2d =====
BipedalWalker-v3       BipedalWalkerHardcore-v3 CarRacing-v3
LunarLander-v3         LunarLanderContinuous-v3
===== toy_text =====
Blackjack-v1           CliffWalking-v1        CliffWalkingSlippery-v1
FrozenLake-v1          FrozenLake8x8-v1       Taxi-v4
===== tabular =====
tabular/Blackjack-v0   tabular/CliffWalking-v0
===== None =====
Ant-v2                 Ant-v3                 GymV21Environment-v0
GymV26Environment-v0   HalfCheetah-v2         HalfCheetah-v3
Hopper-v2              Hopper-v3              Humanoid-v2
Humanoid-v3            HumanoidStandup-v2     InvertedDoublePendulum-v2
InvertedPendulum-v2    Pusher-v2              Reacher-v2
Swimmer-v2             Swimmer-v3             Walker2d-v2
Walker2d-v3
===== mujoco ====

In [47]:
env = gym.make("gymnasium_env/GridWorld-v0", size=5)
print(env)

# Access the unwrapped instance to reach your custom attributes directly
print("Grid size:", env.unwrapped.size)

<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridWorldEnv<gymnasium_env/GridWorld-v0>>>>>
Grid size: 5


In [48]:
env = gym.make("gymnasium_env/GridWorld-v0", size=5, render_mode="human")

obs, info = env.reset(seed=42)
print("Start:", obs, info)

total_reward = 0
for t in range(30):
    action = env.action_space.sample()  # random policy — swap this for your RL agent
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    print(f"step {t:2d} | action={action} | obs={obs} | reward={reward} | terminated={terminated}")

    if terminated or truncated:
        print(f"\nEpisode finished after {t+1} steps. Total reward: {total_reward}")
        break

env.unwrapped.render()
env.close()

Start: {'agent': array([0, 3]), 'target': array([3, 2])} {'distance': np.float64(4.0)}
step  0 | action=0 | obs={'agent': array([0, 4]), 'target': array([3, 2])} | reward=0 | terminated=False
step  1 | action=3 | obs={'agent': array([1, 4]), 'target': array([3, 2])} | reward=0 | terminated=False
step  2 | action=2 | obs={'agent': array([1, 3]), 'target': array([3, 2])} | reward=0 | terminated=False
step  3 | action=1 | obs={'agent': array([0, 3]), 'target': array([3, 2])} | reward=0 | terminated=False
step  4 | action=2 | obs={'agent': array([0, 2]), 'target': array([3, 2])} | reward=0 | terminated=False
step  5 | action=0 | obs={'agent': array([0, 3]), 'target': array([3, 2])} | reward=0 | terminated=False
step  6 | action=2 | obs={'agent': array([0, 2]), 'target': array([3, 2])} | reward=0 | terminated=False
step  7 | action=2 | obs={'agent': array([0, 1]), 'target': array([3, 2])} | reward=0 | terminated=False
step  8 | action=0 | obs={'agent': array([0, 2]), 'target': array([3, 2])

In [49]:
from gymnasium.wrappers import FlattenObservation

env = gym.make("gymnasium_env/GridWorld-v0")
print("Original observation_space:", env.observation_space)

wrapped_env = FlattenObservation(env)
print("Flattened observation_space:", wrapped_env.observation_space)

obs, info = wrapped_env.reset(seed=0)
print("Flattened obs (agent_x, agent_y, target_x, target_y):", obs)

Original observation_space: Dict('agent': Box(0, 4, (2,), int64), 'target': Box(0, 4, (2,), int64))
Flattened observation_space: Box(0, 4, (4,), int64)
Flattened obs (agent_x, agent_y, target_x, target_y): [4 3 2 1]


 Making the environment actually pop up a window (pygame)

In [50]:
import pygame

class GridWorldEnv(gym.Env):
    # rgb_array added alongside human — lets this work headless too
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 4}

    def __init__(self, size: int = 10, render_mode: Optional[str] = None):
        self.size = size
        self.window_size = 512  # pixels, for the pygame window
        self.render_mode = render_mode

        self._agent_location = np.array([-1, -1], dtype=np.int32)
        self._target_location = np.array([-1, -1], dtype=np.int32)

        self.observation_space = gym.spaces.Dict(
            {
                "agent": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),
                "target": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),
            }
        )
        self.action_space = gym.spaces.Discrete(4)
        self._action_to_direction = {
            0: np.array([0, 1]),
            1: np.array([-1, 0]),
            2: np.array([0, -1]),
            3: np.array([1, 0]),
        }

        # Window/clock created lazily on first render — keeps headless
        # training (render_mode=None) completely free of any pygame calls
        self.window = None
        self.clock = None

    def _get_obs(self):
        return {"agent": self._agent_location, "target": self._target_location}

    def _get_info(self):
        return {"distance": np.linalg.norm(self._agent_location - self._target_location, ord=1)}

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        self._agent_location = self.np_random.integers(0, self.size, size=2, dtype=int)
        self._target_location = self._agent_location
        while np.array_equal(self._target_location, self._agent_location):
            self._target_location = self.np_random.integers(0, self.size, size=2, dtype=int)

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()  # draw the window right away so it shows the start state

        return observation, info

    def step(self, action):
        direction = self._action_to_direction[action]
        self._agent_location = np.clip(self._agent_location + direction, 0, self.size - 1)
        terminated = np.array_equal(self._agent_location, self._target_location)
        truncated = False
        reward = 1 if terminated else 0
        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()  # redraw after every step -> live animation

        return observation, reward, terminated, truncated, info

    def render(self):
        # call this yourself when render_mode="rgb_array" (human mode renders automatically)
        if self.render_mode == "rgb_array":
            return self._render_frame()

    def _render_frame(self):
        if self.window is None and self.render_mode == "human":
            pygame.init()
            pygame.display.init()
            self.window = pygame.display.set_mode((self.window_size, self.window_size))
            pygame.display.set_caption("GridWorld")
        if self.clock is None and self.render_mode == "human":
            self.clock = pygame.time.Clock()

        canvas = pygame.Surface((self.window_size, self.window_size))
        canvas.fill((255, 255, 255))
        pix_square_size = self.window_size / self.size

        # target: red square
        pygame.draw.rect(
            canvas,
            (255, 0, 0),
            pygame.Rect(
                pix_square_size * self._target_location,
                (pix_square_size, pix_square_size),
            ),
        )
        # agent: blue circle
        pygame.draw.circle(
            canvas,
            (0, 0, 255),
            (self._agent_location + 0.5) * pix_square_size,
            pix_square_size / 3,
        )
        # gridlines
        for x in range(self.size + 1):
            pygame.draw.line(canvas, 0, (0, pix_square_size * x), (self.window_size, pix_square_size * x), width=3)
            pygame.draw.line(canvas, 0, (pix_square_size * x, 0), (pix_square_size * x, self.window_size), width=3)

        if self.render_mode == "human":
            self.window.blit(canvas, canvas.get_rect())
            pygame.event.pump()
            pygame.display.update()
            self.clock.tick(self.metadata["render_fps"])  # caps the framerate
        else:  # rgb_array
            return np.transpose(np.array(pygame.surfarray.pixels3d(canvas)), axes=(1, 0, 2))

    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()

In [51]:
# Overwrite the previous registration so gym.make() uses our new pygame-enabled class
if "gymnasium_env/GridWorld-v0" in gym.registry:
    del gym.registry["gymnasium_env/GridWorld-v0"]

gym.register(
    id="gymnasium_env/GridWorld-v0",
    entry_point=GridWorldEnv,
    max_episode_steps=300,
)
print("Re-registered with pygame rendering support")

Re-registered with pygame rendering support


In [54]:
env = gym.make("gymnasium_env/GridWorld-v0", size=12, render_mode="human", wrap='cylindrical')
import time  

obs, info = env.reset(seed=7)
print("Start:", obs, info)

# Demonstrate wrapping: if agent moves right off the edge it should reappear at x=0
# Run until we see wrapping or target reached
for t in range(60):
    action = env.action_space.sample()  # swap for your trained policy
    obs_before = obs.copy()
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"step {t:2d} | action={action} | before={obs_before['agent']} -> after={obs['agent']} | reward={reward} | terminated={terminated}")
    time.sleep(0.15)
    if terminated or truncated:
        print(f"Reached target in {t+1} steps!")
        time.sleep(1)
        break

env.close()  # always close the pygame window when done


TypeError: GridWorldEnv.__init__() got an unexpected keyword argument 'wrap' was raised from the environment creator for gymnasium_env/GridWorld-v0 with kwargs ({'size': 12, 'render_mode': 'human', 'wrap': 'cylindrical'})

Adding Fading Trail

In [55]:
import pygame

class GridWorldEnv(gym.Env):
    # rgb_array added alongside human — lets this work headless too
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 4}

    def __init__(self, size: int = 10, render_mode: Optional[str] = None, wrap: str = "none"):
        self.size = size
        self.window_size = 512  # pixels, for the pygame window
        self.render_mode = render_mode
        self.wrap = (wrap or "none").lower()
        self._trail = []  # store agent's path for rendering
        self._max_trail = 40  # keep trail reasonably short for performance

        self._agent_location = np.array([-1, -1], dtype=np.int32)
        self._target_location = np.array([-1, -1], dtype=np.int32)

        self.observation_space = gym.spaces.Dict(
            {
                "agent": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),
                "target": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),
            }
        )
        self.action_space = gym.spaces.Discrete(4)
        self._action_to_direction = {
            0: np.array([0, 1]),
            1: np.array([-1, 0]),
            2: np.array([0, -1]),
            3: np.array([1, 0]),
        }

        # Window/clock created lazily on first render — keeps headless
        # training (render_mode=None) completely free of any pygame calls
        self.window = None
        self.clock = None

    def _get_obs(self):
        return {"agent": self._agent_location, "target": self._target_location}

    def _get_info(self):
        return {"distance": np.linalg.norm(self._agent_location - self._target_location, ord=1)}

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        self._agent_location = self.np_random.integers(0, self.size, size=2, dtype=int)
        self._target_location = self._agent_location
        self._trail = [self._agent_location.copy()]  # reset trail to start with agent's initial position
        while np.array_equal(self._target_location, self._agent_location):
            self._target_location = self.np_random.integers(0, self.size, size=2, dtype=int)

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()  # draw the window right away so it shows the start state

        return observation, info

    def step(self, action):
        direction = self._action_to_direction[action]
        # compute new location according to wrap mode
        candidate = self._agent_location + direction

        if self.wrap == "cylindrical":
            # wrap X (columns), clamp Y (rows)
            new_x = int(candidate[0] % self.size)
            new_y = int(np.clip(candidate[1], 0, self.size - 1))
            self._agent_location = np.array([new_x, new_y], dtype=int)
        elif self.wrap == "toroidal":
            # wrap both axes
            self._agent_location = np.mod(candidate, self.size).astype(int)
        else:
            # no wrapping — clamp both axes
            self._agent_location = np.clip(candidate, 0, self.size - 1).astype(int)

        self._trail.append(self._agent_location.copy())  # add current position to trail
        # keep only the most recent positions
        if len(self._trail) > self._max_trail:
            self._trail = self._trail[-self._max_trail:]

        terminated = np.array_equal(self._agent_location, self._target_location)
        truncated = False
        reward = 1 if terminated else 0
        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()  # redraw after every step -> live animation

        return observation, reward, terminated, truncated, info

    def render(self):
        # call this yourself when render_mode="rgb_array" (human mode renders automatically)
        if self.render_mode == "rgb_array":
            return self._render_frame()

    def _render_frame(self):
        if self.window is None and self.render_mode == "human":
            pygame.init()
            pygame.display.init()
            self.window = pygame.display.set_mode((self.window_size, self.window_size))
            pygame.display.set_caption("GridWorld")
        if self.clock is None and self.render_mode == "human":
            self.clock = pygame.time.Clock()

        canvas = pygame.Surface((self.window_size, self.window_size))
        canvas.fill((255, 255, 255))
        pix_square_size = self.window_size / self.size

        # target: red square
        pygame.draw.rect(
            canvas,
            (255, 0, 0),
            pygame.Rect(
                pix_square_size * self._target_location,
                (pix_square_size, pix_square_size),
            ),
        )
        # agent: blue circle
        pygame.draw.circle(
            canvas,
            (0, 0, 255),
            (int((self._agent_location[0] + 0.5) * pix_square_size), int((self._agent_location[1] + 0.5) * pix_square_size)),
            int(pix_square_size / 3),
        )
        # gridlines
        for x in range(self.size + 1):
            pygame.draw.line(canvas, 0, (0, pix_square_size * x), (self.window_size, pix_square_size * x), width=3)
            pygame.draw.line(canvas, 0, (pix_square_size * x, 0), (pix_square_size * x, self.window_size), width=3)

        # Draw the trail of the agent's path BEFORE blit/return so it is visible
        trail = self._trail[-self._max_trail:]
        trail_length = len(trail)
        if trail_length > 0:
            for idx, location in enumerate(trail):
                # idx=0 oldest, idx=trail_length-1 newest
                # brightness: older -> nearer to white, newer -> full blue
                brightness = 0.25 + 0.75 * ((idx + 1) / trail_length)  # in [0.25,1.0]
                r = int(255 * (1 - brightness))
                g = int(255 * (1 - brightness))
                b = 255
                color = (r, g, b)
                center = (int((location[0] + 0.5) * pix_square_size), int((location[1] + 0.5) * pix_square_size))
                # Make newer points slightly larger for visibility
                radius = int(max(1, (pix_square_size / 6) * (0.5 + 0.5 * ((idx + 1) / trail_length))))
                pygame.draw.circle(canvas, color, center, radius)

        if self.render_mode == "human":
            self.window.blit(canvas, canvas.get_rect())
            pygame.event.pump()
            pygame.display.update()
            self.clock.tick(self.metadata["render_fps"])  # caps the framerate
        else:  # rgb_array
            return np.transpose(np.array(pygame.surfarray.pixels3d(canvas)), axes=(1, 0, 2))

    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()


In [56]:
# Overwrite the previous registration so gym.make() uses our new pygame-enabled class
if "gymnasium_env/GridWorld-v0" in gym.registry:
    del gym.registry["gymnasium_env/GridWorld-v0"]

gym.register(
    id="gymnasium_env/GridWorld-v0",
    entry_point=GridWorldEnv,
    max_episode_steps=300,
)
print("Re-registered with pygame rendering support")

Re-registered with pygame rendering support


In [58]:
env = gym.make("gymnasium_env/GridWorld-v0", size=5, render_mode="human")
import time  

obs, info = env.reset(seed=7)
for t in range(20):
    action = env.action_space.sample()  # swap for your trained policy
    obs, reward, terminated, truncated, info = env.step(action)
    time.sleep(0.5)
    if terminated or truncated:
        print(f"Reached target in {t+1} steps!")
        time.sleep(3)
        break

env.close()  # always close the pygame window when done

Displaying Reinforcement Rewards

In [ ]:
import pygame

class GridWorldEnv(gym.Env):
    # rgb_array added alongside human — lets this work headless too
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 4}

    def __init__(self, size: int = 10, render_mode: Optional[str] = None, wrap: str = "none"):
        self.size = size
        self.window_size = 512  # pixels for the grid area
        self.header_height = 40  # pixels for the display above the grid
        self.render_mode = render_mode
        self.wrap = (wrap or "none").lower()
        self._trail = []  # store agent's path for rendering
        self._max_trail = 40  # keep trail reasonably short for performance

        self._agent_location = np.array([-1, -1], dtype=np.int32)
        self._target_location = np.array([-1, -1], dtype=np.int32)

        # reward bookkeeping for display
        self.cumulative_reward = 0
        self._last_reward = 0

        self.observation_space = gym.spaces.Dict(
            {
                "agent": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),
                "target": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),
            }
        )
        self.action_space = gym.spaces.Discrete(4)
        self._action_to_direction = {
            0: np.array([0, 1]),
            1: np.array([-1, 0]),
            2: np.array([0, -1]),
            3: np.array([1, 0]),
        }

        # Window/clock created lazily on first render — keeps headless
        # training (render_mode=None) completely free of any pygame calls
        self.window = None
        self.clock = None

    def _get_obs(self):
        return {"agent": self._agent_location, "target": self._target_location}

    def _get_info(self):
        return {"distance": np.linalg.norm(self._agent_location - self._target_location, ord=1)}

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        self._agent_location = self.np_random.integers(0, self.size, size=2, dtype=int)
        self._target_location = self._agent_location
        self._trail = [self._agent_location.copy()]  # reset trail to start with agent's initial position
        self.cumulative_reward = 0
        self._last_reward = 0
        while np.array_equal(self._target_location, self._agent_location):
            self._target_location = self.np_random.integers(0, self.size, size=2, dtype=int)

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()  # draw the window right away so it shows the start state

        return observation, info

    def step(self, action):
        direction = self._action_to_direction[action]
        # compute new location according to wrap mode
        candidate = self._agent_location + direction

        if self.wrap == "cylindrical":
            # wrap X (columns), clamp Y (rows)
            new_x = int(candidate[0] % self.size)
            new_y = int(np.clip(candidate[1], 0, self.size - 1))
            self._agent_location = np.array([new_x, new_y], dtype=int)
        elif self.wrap == "toroidal":
            # wrap both axes
            self._agent_location = np.mod(candidate, self.size).astype(int)
        else:
            # no wrapping — clamp both axes
            self._agent_location = np.clip(candidate, 0, self.size - 1).astype(int)

        self._trail.append(self._agent_location.copy())  # add current position to trail
        # keep only the most recent positions
        if len(self._trail) > self._max_trail:
            self._trail = self._trail[-self._max_trail:]

        terminated = np.array_equal(self._agent_location, self._target_location)
        truncated = False

        # reward logic (current): +1 for reaching target, 0 otherwise
        # this is where you can add shaping (per-step punishment, distance-based etc.)
        reward = 1 if terminated else 0

        # bookkeeping for display
        self._last_reward = reward
        self.cumulative_reward += reward

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()  # redraw after every step -> live animation

        return observation, reward, terminated, truncated, info

    def render(self):
        # call this yourself when render_mode="rgb_array" (human mode renders automatically)
        if self.render_mode == "rgb_array":
            return self._render_frame()

    def _render_frame(self):
        # create window lazily; include header height in window size
        total_height = self.window_size + self.header_height
        if self.window is None and self.render_mode == "human":
            pygame.init()
            pygame.display.init()
            pygame.font.init()
            self.window = pygame.display.set_mode((self.window_size, total_height))
            pygame.display.set_caption("GridWorld")
        if self.clock is None and self.render_mode == "human":
            self.clock = pygame.time.Clock()

        canvas = pygame.Surface((self.window_size, total_height))
        canvas.fill((255, 255, 255))

        # Draw reward header in its own area above the grid
        try:
            font = pygame.font.SysFont(None, 20)
            header_rect = pygame.Rect(0, 0, self.window_size, self.header_height)
            pygame.draw.rect(canvas, (245, 245, 245), header_rect)
            pos = tuple(int(x) for x in self._agent_location)
            text = f"Last: {self._last_reward}   Total: {self.cumulative_reward}   Pos: {pos}"
            text_surf = font.render(text, True, (0, 0, 0))
            canvas.blit(text_surf, (8, 8))
        except Exception:
            pass

        # grid origin (top-left) starts after the header
        grid_origin_y = self.header_height
        pix_square_size = self.window_size / self.size

        # target: red square (adjusted for header offset)
        tx = int(self._target_location[0] * pix_square_size)
        ty = int(grid_origin_y + self._target_location[1] * pix_square_size)
        tw = int(pix_square_size)
        th = int(pix_square_size)
        pygame.draw.rect(canvas, (255, 0, 0), pygame.Rect(tx, ty, tw, th))

        # agent: blue circle (adjusted for header offset)
        agent_center = (
            int((self._agent_location[0] + 0.5) * pix_square_size),
            int(grid_origin_y + (self._agent_location[1] + 0.5) * pix_square_size),
        )
        pygame.draw.circle(canvas, (0, 0, 255), agent_center, int(pix_square_size / 3))

        # gridlines (offset vertically by header)
        for x in range(self.size + 1):
            y = int(grid_origin_y + pix_square_size * x)
            pygame.draw.line(canvas, 0, (0, y), (self.window_size, y), width=1)
            pygame.draw.line(canvas, 0, (int(pix_square_size * x), grid_origin_y), (int(pix_square_size * x), grid_origin_y + self.window_size), width=1)

        # Draw the trail of the agent's path BEFORE blit/return so it is visible
        trail = self._trail[-self._max_trail:]
        trail_length = len(trail)
        if trail_length > 0:
            for idx, location in enumerate(trail):
                # idx=0 oldest, idx=trail_length-1 newest
                brightness = 0.25 + 0.75 * ((idx + 1) / trail_length)  # in [0.25,1.0]
                r = int(255 * (1 - brightness))
                g = int(255 * (1 - brightness))
                b = 255
                color = (r, g, b)
                center = (
                    int((location[0] + 0.5) * pix_square_size),
                    int(grid_origin_y + (location[1] + 0.5) * pix_square_size),
                )
                radius = int(max(1, (pix_square_size / 6) * (0.5 + 0.5 * ((idx + 1) / trail_length))))
                pygame.draw.circle(canvas, color, center, radius)

        if self.render_mode == "human":
            self.window.blit(canvas, canvas.get_rect())
            pygame.event.pump()
            pygame.display.update()
            self.clock.tick(self.metadata["render_fps"])  # caps the framerate
        else:  # rgb_array
            return np.transpose(np.array(pygame.surfarray.pixels3d(canvas)), axes=(1, 0, 2))

    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()


In [ ]:
# Overwrite the previous registration so gym.make() uses our new pygame-enabled class
if "gymnasium_env/GridWorld-v0" in gym.registry:
    del gym.registry["gymnasium_env/GridWorld-v0"]

gym.register(
    id="gymnasium_env/GridWorld-v0",
    entry_point=GridWorldEnv,
    max_episode_steps=300,
)
print("Re-registered with pygame rendering support")

Re-registered with pygame rendering support


In [ ]:
env = gym.make(
    "gymnasium_env/GridWorld-v0",
    size=20,
    render_mode="human",
    wrap='toroidal',
    step_penalty=-0.02,
    shaping=True,
    shaping_coeff=2.0,
    away_coeff=0.4,
    success_reward=1.0,
)
import time

obs, info = env.reset(seed=7)
print("Start:", obs, info)

# Run a longer random policy to see shaping behavior on a larger grid
total_reward = 0.0
for t in range(120):
    action = env.action_space.sample()  # swap for your trained policy
    obs_before = obs.copy()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    print(f"step {t:3d} | action={action} | before={tuple(int(x) for x in obs_before['agent'])} -> after={tuple(int(x) for x in obs['agent'])} | reward={reward:.3f} | terminated={terminated}")
    time.sleep(0.12)
    if terminated or truncated:
        print(f"Reached target in {t+1} steps! Total reward: {total_reward:.3f}")
        time.sleep(1)
        break

env.close()  # always close the pygame window when done


Adding Reward

In [ ]:
import pygame

class GridWorldEnv(gym.Env):
    # rgb_array added alongside human — lets this work headless too
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 4}

    def __init__(self,
                 size: int = 10,
                 render_mode: Optional[str] = None,
                 wrap: str = "none",
                 step_penalty: float = -0.01,
                 shaping: bool = True,
                 shaping_coeff: float = 1.0,
                 away_coeff: float = 0.5,
                 success_reward: float = 1.0):
        """
        GridWorld with optional distance-based reward shaping.

        Parameters:
        - size: grid size (size x size)
        - render_mode: 'human' or 'rgb_array'
        - wrap: 'none', 'cylindrical', or 'toroidal'
        - step_penalty: small per-step penalty to encourage shorter trajectories
        - shaping: enable distance-based shaping
        - shaping_coeff: coefficient for positive progress (closer -> positive)
        - away_coeff: scale for negative progress (moving away). Should be < 1 to punish less than reward.
        - success_reward: reward when reaching the target
        """
        self.size = size
        self.window_size = 512  # pixels for the grid area
        self.header_height = 40  # pixels for the display above the grid
        self.render_mode = render_mode
        self.wrap = (wrap or "none").lower()
        self._trail = []  # store agent's path for rendering
        self._max_trail = 80  # keep trail reasonably short for performance

        # reward shaping parameters
        self.step_penalty = float(step_penalty)
        self.shaping = bool(shaping)
        self.shaping_coeff = float(shaping_coeff)
        self.away_coeff = float(away_coeff)
        self.success_reward = float(success_reward)

        self._agent_location = np.array([-1, -1], dtype=np.int32)
        self._target_location = np.array([-1, -1], dtype=np.int32)

        # reward bookkeeping for display
        self.cumulative_reward = 0.0
        self._last_reward = 0.0

        self.observation_space = gym.spaces.Dict(
            {
                "agent": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),
                "target": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),
            }
        )
        self.action_space = gym.spaces.Discrete(4)
        self._action_to_direction = {
            0: np.array([0, 1]),
            1: np.array([-1, 0]),
            2: np.array([0, -1]),
            3: np.array([1, 0]),
        }

        # Window/clock created lazily on first render — keeps headless
        # training (render_mode=None) completely free of any pygame calls
        self.window = None
        self.clock = None

    def _get_obs(self):
        return {"agent": self._agent_location, "target": self._target_location}

    def _get_info(self):
        # include both Manhattan and Euclidean distances for diagnostics
        manhattan = np.linalg.norm(self._agent_location - self._target_location, ord=1)
        euclidean = float(np.linalg.norm(self._agent_location - self._target_location, ord=2))
        return {"manhattan": manhattan, "euclidean": euclidean}

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        self._agent_location = self.np_random.integers(0, self.size, size=2, dtype=int)
        self._target_location = self._agent_location
        self._trail = [self._agent_location.copy()]  # reset trail to start with agent's initial position
        self.cumulative_reward = 0.0
        self._last_reward = 0.0
        while np.array_equal(self._target_location, self._agent_location):
            self._target_location = self.np_random.integers(0, self.size, size=2, dtype=int)

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()  # draw the window right away so it shows the start state

        return observation, info

    def step(self, action):
        # compute previous distance (Euclidean) before moving
        prev_dist = float(np.linalg.norm(self._agent_location - self._target_location, ord=2))

        direction = self._action_to_direction[action]
        # compute new location according to wrap mode
        candidate = self._agent_location + direction

        if self.wrap == "cylindrical":
            # wrap X (columns), clamp Y (rows)
            new_x = int(candidate[0] % self.size)
            new_y = int(np.clip(candidate[1], 0, self.size - 1))
            self._agent_location = np.array([new_x, new_y], dtype=int)
        elif self.wrap == "toroidal":
            # wrap both axes
            self._agent_location = np.mod(candidate, self.size).astype(int)
        else:
            # no wrapping — clamp both axes
            self._agent_location = np.clip(candidate, 0, self.size - 1).astype(int)

        self._trail.append(self._agent_location.copy())  # add current position to trail
        # keep only the most recent positions
        if len(self._trail) > self._max_trail:
            self._trail = self._trail[-self._max_trail:]

        terminated = np.array_equal(self._agent_location, self._target_location)
        truncated = False

        # compute new distance and delta (positive when closer)
        new_dist = float(np.linalg.norm(self._agent_location - self._target_location, ord=2))
        delta = prev_dist - new_dist

        # base reward: success reward if terminated, otherwise apply step penalty
        if terminated:
            reward = float(self.success_reward)
        else:
            reward = float(self.step_penalty)
            if self.shaping:
                if delta > 0:
                    reward += self.shaping_coeff * float(delta)
                elif delta < 0:
                    # moving away: scale punishment so it's smaller in magnitude
                    reward += self.shaping_coeff * self.away_coeff * float(delta)  # delta is negative

        # bookkeeping for display
        self._last_reward = float(reward)
        self.cumulative_reward += float(reward)

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()  # redraw after every step -> live animation

        return observation, reward, terminated, truncated, info

    def render(self):
        # call this yourself when render_mode="rgb_array" (human mode renders automatically)
        if self.render_mode == "rgb_array":
            return self._render_frame()

    def _render_frame(self):
        # create window lazily; include header height in window size
        total_height = self.window_size + self.header_height
        if self.window is None and self.render_mode == "human":
            pygame.init()
            pygame.display.init()
            pygame.font.init()
            self.window = pygame.display.set_mode((self.window_size, total_height))
            pygame.display.set_caption("GridWorld")
        if self.clock is None and self.render_mode == "human":
            self.clock = pygame.time.Clock()

        canvas = pygame.Surface((self.window_size, total_height))
        canvas.fill((255, 255, 255))

        # Draw reward header in its own area above the grid
        try:
            font = pygame.font.SysFont(None, 20)
            header_rect = pygame.Rect(0, 0, self.window_size, self.header_height)
            pygame.draw.rect(canvas, (245, 245, 245), header_rect)
            pos = tuple(int(x) for x in self._agent_location)
            text = (
                f"Last: {self._last_reward:.3f}   Total: {self.cumulative_reward:.3f}   Pos: {pos}"
                f"   step_penalty={self.step_penalty} shaping={self.shaping} s_coeff={self.shaping_coeff} a_coeff={self.away_coeff}"
            )
            text_surf = font.render(text, True, (0, 0, 0))
            canvas.blit(text_surf, (8, 8))
        except Exception:
            pass

        # grid origin (top-left) starts after the header
        grid_origin_y = self.header_height
        pix_square_size = self.window_size / self.size

        # target: red square (adjusted for header offset)
        tx = int(self._target_location[0] * pix_square_size)
        ty = int(grid_origin_y + self._target_location[1] * pix_square_size)
        tw = int(pix_square_size)
        th = int(pix_square_size)
        pygame.draw.rect(canvas, (255, 0, 0), pygame.Rect(tx, ty, tw, th))

        # agent: blue circle (adjusted for header offset)
        agent_center = (
            int((self._agent_location[0] + 0.5) * pix_square_size),
            int(grid_origin_y + (self._agent_location[1] + 0.5) * pix_square_size),
        )
        pygame.draw.circle(canvas, (0, 0, 255), agent_center, int(pix_square_size / 3))

        # gridlines (offset vertically by header)
        for x in range(self.size + 1):
            y = int(grid_origin_y + pix_square_size * x)
            pygame.draw.line(canvas, 0, (0, y), (self.window_size, y), width=1)
            pygame.draw.line(canvas, 0, (int(pix_square_size * x), grid_origin_y), (int(pix_square_size * x), grid_origin_y + self.window_size), width=1)

        # Draw the trail of the agent's path BEFORE blit/return so it is visible
        trail = self._trail[-self._max_trail:]
        trail_length = len(trail)
        if trail_length > 0:
            for idx, location in enumerate(trail):
                # idx=0 oldest, idx=trail_length-1 newest
                brightness = 0.25 + 0.75 * ((idx + 1) / trail_length)  # in [0.25,1.0]
                r = int(255 * (1 - brightness))
                g = int(255 * (1 - brightness))
                b = 255
                color = (r, g, b)
                center = (
                    int((location[0] + 0.5) * pix_square_size),
                    int(grid_origin_y + (location[1] + 0.5) * pix_square_size),
                )
                radius = int(max(1, (pix_square_size / 6) * (0.5 + 0.5 * ((idx + 1) / trail_length))))
                pygame.draw.circle(canvas, color, center, radius)

        if self.render_mode == "human":
            self.window.blit(canvas, canvas.get_rect())
            pygame.event.pump()
            pygame.display.update()
            self.clock.tick(self.metadata["render_fps"])  # caps the framerate
        else:  # rgb_array
            return np.transpose(np.array(pygame.surfarray.pixels3d(canvas)), axes=(1, 0, 2))

    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()


In [ ]:
# Overwrite the previous registration so gym.make() uses our new pygame-enabled class
if "gymnasium_env/GridWorld-v0" in gym.registry:
    del gym.registry["gymnasium_env/GridWorld-v0"]

gym.register(
    id="gymnasium_env/GridWorld-v0",
    entry_point=GridWorldEnv,
    max_episode_steps=300,
)
print("Re-registered with pygame rendering support")

Re-registered with pygame rendering support


In [ ]:
env = gym.make("gymnasium_env/GridWorld-v0", size=5, render_mode="human")
import time  
import random

obs, info = env.reset(seed=random.randint(0, 10000))
for t in range(50):
    action = env.action_space.sample()  # swap for your trained policy
    obs, reward, terminated, truncated, info = env.step(action)
    time.sleep(0.5)
    if terminated or truncated:
        print(f"Reached target in {t+1} steps!")
        time.sleep(3)
        break

env.close()  # always close the pygame window when done